# Sinhala Buddhist Corpus Builder - CNN Filtered + Document AI Batch

This notebook combines:
1. **CNN Page Classification** - Filter out non-content pages (covers, blank pages, etc.)
2. **PDF Filtering** - Create new PDFs with only content pages
3. **Document AI Batch Processing** - Process only the filtered content pages

**Key Benefits:**
- Save money by not processing non-content pages
- Higher quality corpus (only actual content)
- Detailed statistics on filtering effectiveness
- Full checkpoint/resume capability

## 1. Installation and Setup

In [ ]:
# Install required packages with compatible versions for Colab
# Note: We avoid upgrading google-auth and google-cloud-storage to prevent conflicts
!pip install google-cloud-documentai PyMuPDF Pillow -q
# tqdm, torch, and torchvision are already installed in Colab

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check package versions and compatibility
import pkg_resources

packages_to_check = [
    'google-cloud-documentai',
    'google-cloud-storage',
    'google-auth',
    'torch',
    'torchvision'
]

print("Installed package versions:")
for package in packages_to_check:
    try:
        version = pkg_resources.get_distribution(package).version
        print(f"  {package}: {version}")
    except:
        print(f"  {package}: Not installed")

print("\n⚠️ Note: Some dependency warnings are expected in Colab and can be safely ignored.")
print("   The code will work correctly despite version conflicts.")

## 2. Import Required Libraries

In [ ]:
import os
import json
import time
import re
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import fitz  # PyMuPDF
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms

from google.cloud import documentai_v1 as documentai
from google.cloud import storage
from google.oauth2 import service_account
from google.api_core.client_options import ClientOptions
from tqdm.notebook import tqdm
import pandas as pd

print("✅ All imports successful!")

## 3. Configuration

In [ ]:
# Base directory configuration
BASE_DIR = Path("/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/")
PDF_FOLDER = BASE_DIR / "data" / "raw" / "old_books"

PAGE_CLASSIFIER_MODEL_PATH = BASE_DIR / "models" / "extra" / "page_classifier_kfold" / "best_fold_model.pth"

# Document AI extraction folders
EXTRACTION_BASE = BASE_DIR / "data" / "docai_extractions"
RAW_TEXT_DIR = EXTRACTION_BASE / "1_raw_text"
CLEANED_TEXT_DIR = EXTRACTION_BASE / "2_cleaned_text"

# Filtered PDFs (CNN filtered, ready for Document AI)
FILTERED_PDFS_DIR = BASE_DIR / "data" / "filtered_pdfs"

# Logs and metadata
LOGS_DIR = EXTRACTION_BASE / "logs"
CHECKPOINT_FILE = LOGS_DIR / "processing_checkpoint_filtered.json"
FILTERING_STATS_FILE = LOGS_DIR / "filtering_stats.json"

# Create all directories
for directory in [RAW_TEXT_DIR, CLEANED_TEXT_DIR, LOGS_DIR, FILTERED_PDFS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("✓ Directory structure created")
print(f"  PDFs from: {PDF_FOLDER}")
print(f"  CNN Model: {PAGE_CLASSIFIER_MODEL_PATH}")
print(f"  Filtered PDFs: {FILTERED_PDFS_DIR}")
print(f"  Raw text to: {RAW_TEXT_DIR}")
print(f"  Cleaned text to: {CLEANED_TEXT_DIR}")
print(f"  Logs to: {LOGS_DIR}")

In [ ]:
# Google Cloud Authentication
CREDENTIALS_PATH = BASE_DIR / "auth" / "Vision OCR thematic runner.json"

if not CREDENTIALS_PATH.exists():
    print("⚠️ WARNING: Credentials file not found!")
    print(f"   Looking for: {CREDENTIALS_PATH}")
else:
    print("✓ Credentials file found")

In [ ]:
# Google Cloud Configuration
# TODO: Update these with your actual values
PROJECT_ID = "thematic-runner-453210-s8"  # e.g., "buddhist-nlp-project"
LOCATION = "us"  # e.g., "us" or "eu"
PROCESSOR_ID = "11b17ff9bb09b5db"  # e.g., "abc123def456"

# GCS Configuration
GCS_BUCKET_NAME = f"{PROJECT_ID}-docai-batch"  # Will be auto-created
GCS_INPUT_PREFIX = "input_pdfs/"  # Folder for PDFs
GCS_OUTPUT_PREFIX = "output_results/"  # Folder for results

# Processing configuration
POLL_INTERVAL = 30
MAX_POLL_ATTEMPTS = 120
RETRY_ATTEMPTS = 3

# CNN Classifier configuration
CNN_CONFIDENCE_THRESHOLD = 0.5  # Minimum confidence to classify as content

print("Configuration:")
print(f"  Project: {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Processor: {PROCESSOR_ID}")
print(f"  GCS Bucket: {GCS_BUCKET_NAME}")
print(f"  CNN Threshold: {CNN_CONFIDENCE_THRESHOLD}")

## 3. Load CNN Page Classifier Model

In [ ]:
# CNN Model Architecture (Must match your training)
class ImprovedPageClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.5):
        super(ImprovedPageClassifier, self).__init__()

        # Use MobileNetV2 as backbone
        self.backbone = models.mobilenet_v2(weights=None)

        # Replace classifier (must match training architecture)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.6),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


# Load trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not PAGE_CLASSIFIER_MODEL_PATH.exists():
    print(f"❌ Model not found at: {PAGE_CLASSIFIER_MODEL_PATH}")
    print(f"   Please upload your trained model to this location.")
    page_classifier_model = None
else:
    try:
        # Load checkpoint
        checkpoint = torch.load(PAGE_CLASSIFIER_MODEL_PATH, map_location=device)

        # Create model
        page_classifier_model = ImprovedPageClassifier()
        page_classifier_model.load_state_dict(checkpoint['model_state_dict'])
        page_classifier_model = page_classifier_model.to(device)
        page_classifier_model.eval()

        print(f"✅ Page classifier model loaded successfully!")
        print(f"   Device: {device}")
        print(f"   Model accuracy: {checkpoint.get('accuracy', 'N/A')}")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        page_classifier_model = None

# Image preprocessing (Must match training)
page_classifier_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 4. CNN Classification Function

In [ ]:
def classify_page_importance(pdf_path: Path, page_num: int) -> Tuple[bool, float]:
    """
    Classify if a page is important (content) using trained CNN.

    Args:
        pdf_path: Path to PDF file
        page_num: Page number (0-indexed)

    Returns:
        Tuple of (is_content: bool, confidence: float)
    """
    # If model not loaded, assume all pages are content
    if page_classifier_model is None:
        return True, 1.0

    try:
        # Extract page as image
        doc = fitz.open(pdf_path)
        page = doc[page_num]
        pix = page.get_pixmap(dpi=150)

        # Convert to PIL Image
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        doc.close()

        # Preprocess image
        img_tensor = page_classifier_transform(img).unsqueeze(0).to(device)

        # Predict
        with torch.no_grad():
            output = page_classifier_model(img_tensor)
            probs = torch.nn.functional.softmax(output, dim=1)
            prediction = torch.argmax(probs, dim=1).item()
            confidence = probs[0][prediction].item()

        # prediction: 0 = not_important, 1 = important
        is_content = (prediction == 1)

        return is_content, confidence

    except Exception as e:
        # If classification fails, assume content (safe fallback)
        print(f"  Warning: Classification failed for page {page_num}: {e}")
        return True, 0.5


print("✓ CNN classification function defined")

## 5. PDF Filtering Functions

In [ ]:
def filter_pdf_pages(
    input_pdf_path: Path,
    output_pdf_path: Path,
    confidence_threshold: float = 0.5
) -> Dict:
    """
    Filter PDF to keep only content pages based on CNN classification.

    Args:
        input_pdf_path: Path to original PDF
        output_pdf_path: Path to save filtered PDF
        confidence_threshold: Minimum confidence to keep page

    Returns:
        Dictionary with filtering statistics
    """
    pdf_name = input_pdf_path.stem

    print(f"\n{'='*70}")
    print(f"🔍 Filtering: {pdf_name}")
    print(f"{'='*70}")

    # Open input PDF
    input_doc = fitz.open(input_pdf_path)
    total_pages = len(input_doc)

    print(f"📄 Total pages: {total_pages}")
    print(f"🤖 Classifying pages with CNN model...\n")

    # Track statistics
    stats = {
        'pdf_name': pdf_name,
        'total_pages': total_pages,
        'content_pages': 0,
        'filtered_pages': 0,
        'content_page_numbers': [],
        'filtered_page_numbers': [],
        'page_classifications': []  # For detailed logging
    }

    # Create output PDF
    output_doc = fitz.open()

    # Classify and filter each page
    for page_num in tqdm(range(total_pages), desc="Classifying pages"):
        is_content, confidence = classify_page_importance(input_pdf_path, page_num)

        # Store classification details
        stats['page_classifications'].append({
            'page_num': page_num + 1,  # 1-indexed for display
            'is_content': is_content,
            'confidence': round(confidence, 4)
        })

        if is_content and confidence >= confidence_threshold:
            # Keep this page
            output_doc.insert_pdf(input_doc, from_page=page_num, to_page=page_num)
            stats['content_pages'] += 1
            stats['content_page_numbers'].append(page_num + 1)
        else:
            # Filter out this page
            stats['filtered_pages'] += 1
            stats['filtered_page_numbers'].append(page_num + 1)

    # Save filtered PDF
    if stats['content_pages'] > 0:
        output_doc.save(output_pdf_path)
        print(f"\n✅ Filtered PDF saved: {output_pdf_path.name}")
    else:
        print(f"\n⚠️  No content pages found - PDF not created")

    # Close documents
    input_doc.close()
    output_doc.close()

    # Print statistics
    print(f"\n📊 Filtering Results:")
    print(f"   Total pages: {stats['total_pages']}")
    print(f"   Content pages: {stats['content_pages']} ({stats['content_pages']/stats['total_pages']*100:.1f}%)")
    print(f"   Filtered pages: {stats['filtered_pages']} ({stats['filtered_pages']/stats['total_pages']*100:.1f}%)")

    if stats['filtered_pages'] > 0 and stats['filtered_pages'] <= 20:
        print(f"   Filtered page numbers: {stats['filtered_page_numbers']}")

    return stats


print("✓ PDF filtering function defined")

## 6. Initialize Google Cloud Clients

In [ ]:
def initialize_clients(credentials_path: Path, project_id: str, location: str):
    """Initialize Document AI and Cloud Storage clients."""
    credentials = service_account.Credentials.from_service_account_file(
        str(credentials_path),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )

    opts = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
    docai_client = documentai.DocumentProcessorServiceClient(
        client_options=opts,
        credentials=credentials
    )

    storage_client = storage.Client(
        credentials=credentials,
        project=project_id
    )

    return docai_client, storage_client, credentials


try:
    docai_client, storage_client, credentials = initialize_clients(
        CREDENTIALS_PATH, PROJECT_ID, LOCATION
    )
    print("✓ Document AI client initialized")
    print("✓ Cloud Storage client initialized")
except Exception as e:
    print(f"✗ Error initializing clients: {e}")

In [ ]:
# Create or get GCS bucket
def create_bucket_if_not_exists(storage_client: storage.Client, bucket_name: str, location: str):
    try:
        bucket = storage_client.get_bucket(bucket_name)
        print(f"✓ Bucket '{bucket_name}' already exists")
    except Exception:
        bucket = storage_client.create_bucket(bucket_name, location=location)
        print(f"✓ Created bucket '{bucket_name}' in {location}")
    return bucket

try:
    bucket = create_bucket_if_not_exists(storage_client, GCS_BUCKET_NAME, LOCATION)
    print(f"\n✓ Using bucket: gs://{GCS_BUCKET_NAME}")
except Exception as e:
    print(f"✗ Error with bucket: {e}")

## 7. Checkpoint Manager

In [ ]:
class CheckpointManager:
    """Manages processing checkpoints for resume capability."""

    def __init__(self, checkpoint_file: Path):
        self.checkpoint_file = checkpoint_file
        self.data = self._load()

    def _load(self) -> dict:
        if self.checkpoint_file.exists():
            with open(self.checkpoint_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        else:
            return {
                "session_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
                "processed_pdfs": {}
            }

    def _save(self):
        with open(self.checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(self.data, f, indent=2, ensure_ascii=False)

    def is_processed(self, pdf_name: str) -> bool:
        return (pdf_name in self.data["processed_pdfs"] and
                self.data["processed_pdfs"][pdf_name].get("status") == "success")

    def mark_filtering_complete(self, pdf_name: str, filtering_stats: dict):
        """Mark PDF filtering as complete."""
        if pdf_name not in self.data["processed_pdfs"]:
            self.data["processed_pdfs"][pdf_name] = {}

        self.data["processed_pdfs"][pdf_name]["filtering_complete"] = True
        self.data["processed_pdfs"][pdf_name]["filtering_stats"] = filtering_stats
        self.data["processed_pdfs"][pdf_name]["filtered_at"] = datetime.now().isoformat()
        self._save()

    def mark_processing(self, pdf_name: str, gcs_input_uri: str, gcs_output_uri: str):
        if pdf_name not in self.data["processed_pdfs"]:
            self.data["processed_pdfs"][pdf_name] = {}

        self.data["processed_pdfs"][pdf_name]["status"] = "processing"
        self.data["processed_pdfs"][pdf_name]["gcs_input_uri"] = gcs_input_uri
        self.data["processed_pdfs"][pdf_name]["gcs_output_uri"] = gcs_output_uri
        self.data["processed_pdfs"][pdf_name]["processing_started"] = datetime.now().isoformat()
        self._save()

    def mark_success(self, pdf_name: str, pages: int):
        if pdf_name in self.data["processed_pdfs"]:
            self.data["processed_pdfs"][pdf_name]["status"] = "success"
            self.data["processed_pdfs"][pdf_name]["extracted_pages"] = pages
            self.data["processed_pdfs"][pdf_name]["completed_at"] = datetime.now().isoformat()
            self._save()

    def mark_failed(self, pdf_name: str, error: str):
        if pdf_name in self.data["processed_pdfs"]:
            self.data["processed_pdfs"][pdf_name]["status"] = "failed"
            self.data["processed_pdfs"][pdf_name]["error"] = error
            self.data["processed_pdfs"][pdf_name]["failed_at"] = datetime.now().isoformat()
            self._save()

    def get_summary(self) -> str:
        processed = self.data["processed_pdfs"]
        total = len(processed)
        success = sum(1 for p in processed.values() if p.get("status") == "success")
        failed = sum(1 for p in processed.values() if p.get("status") == "failed")
        processing = sum(1 for p in processed.values() if p.get("status") == "processing")

        # Filtering stats
        total_original_pages = sum(
            p.get("filtering_stats", {}).get("total_pages", 0)
            for p in processed.values()
        )
        total_content_pages = sum(
            p.get("filtering_stats", {}).get("content_pages", 0)
            for p in processed.values()
        )
        total_filtered_pages = sum(
            p.get("filtering_stats", {}).get("filtered_pages", 0)
            for p in processed.values()
        )

        # Calculate percentages (avoid division by zero)
        if total_original_pages > 0:
            content_pct = total_content_pages / total_original_pages * 100
            filtered_pct = total_filtered_pages / total_original_pages * 100
        else:
            content_pct = 0.0
            filtered_pct = 0.0

        summary = f"""
Checkpoint Summary:
─────────────────────────────────
Total PDFs tracked: {total}
Successfully processed: {success}
Failed: {failed}
Currently processing: {processing}

Filtering Statistics:
  Original pages: {total_original_pages}
  Content pages: {total_content_pages} ({content_pct:.1f}%)
  Filtered out: {total_filtered_pages} ({filtered_pct:.1f}%)

Session ID: {self.data['session_id']}
"""
        return summary


# Initialize checkpoint manager
checkpoint = CheckpointManager(CHECKPOINT_FILE)
print(checkpoint.get_summary())

## 8. GCS and Document AI Functions

In [ ]:
# GCS Functions
def upload_pdf_to_gcs(storage_client, bucket_name, pdf_path, gcs_prefix):
    bucket = storage_client.bucket(bucket_name)
    blob_name = f"{gcs_prefix}{pdf_path.name}"
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(str(pdf_path))
    return f"gs://{bucket_name}/{blob_name}"

def delete_from_gcs(storage_client, gcs_uri):
    if not gcs_uri.startswith("gs://"):
        return
    parts = gcs_uri[5:].split("/", 1)
    bucket_name = parts[0]
    blob_path = parts[1] if len(parts) > 1 else ""
    bucket = storage_client.bucket(bucket_name)

    if blob_path.endswith("/") or not blob_path:
        blobs = list(bucket.list_blobs(prefix=blob_path))
        for blob in blobs:
            blob.delete()
    else:
        blob = bucket.blob(blob_path)
        if blob.exists():
            blob.delete()

def download_results_from_gcs(storage_client, gcs_output_uri):
    parts = gcs_output_uri[5:].split("/", 1)
    bucket_name = parts[0]
    prefix = parts[1] if len(parts) > 1 else ""
    bucket = storage_client.bucket(bucket_name)
    blobs = list(bucket.list_blobs(prefix=prefix))
    json_blobs = [b for b in blobs if b.name.endswith(".json")]
    results = [blob.download_as_bytes() for blob in json_blobs]
    return results

# Document AI Functions
def get_processor_name(project_id, location, processor_id):
    return f"projects/{project_id}/locations/{location}/processors/{processor_id}"

def submit_batch_process_request(client, processor_name, gcs_input_uri, gcs_output_uri):
    gcs_document = documentai.GcsDocument(gcs_uri=gcs_input_uri, mime_type="application/pdf")
    gcs_documents = documentai.GcsDocuments(documents=[gcs_document])
    input_config = documentai.BatchDocumentsInputConfig(gcs_documents=gcs_documents)
    gcs_output_config = documentai.DocumentOutputConfig.GcsOutputConfig(gcs_uri=gcs_output_uri)
    output_config = documentai.DocumentOutputConfig(gcs_output_config=gcs_output_config)
    request = documentai.BatchProcessRequest(
        name=processor_name,
        input_documents=input_config,
        document_output_config=output_config,
    )
    operation = client.batch_process_documents(request)
    return operation

def wait_for_operation(operation, poll_interval=30, max_attempts=120):
    print(f"  Waiting for processing to complete...")
    for attempt in range(max_attempts):
        if operation.done():
            if operation.exception():
                print(f"  ✗ Operation failed: {operation.exception()}")
                return False
            else:
                print(f"  ✓ Operation completed successfully")
                return True
        if attempt % 5 == 0 and attempt > 0:
            elapsed = attempt * poll_interval
            print(f"  Still processing... ({elapsed}s elapsed)")
        time.sleep(poll_interval)
    print(f"  ⚠️ Timeout after {max_attempts * poll_interval}s")
    return False

print("✓ GCS and Document AI functions defined")

## 9. Text Extraction and Cleaning

In [ ]:
def parse_batch_results(result_bytes):
    return documentai.Document.from_json(result_bytes)

def extract_text_by_page(document):
    page_texts = []
    for page in document.pages:
        page_text = ""
        if hasattr(page, 'paragraphs') and page.paragraphs:
            for paragraph in page.paragraphs:
                text_segment = paragraph.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + "\n\n"
        elif hasattr(page, 'lines') and page.lines:
            for line in page.lines:
                text_segment = line.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + "\n"
        elif hasattr(page, 'tokens') and page.tokens:
            for token in page.tokens:
                text_segment = token.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + " "
            page_text = page_text.strip() + "\n"
        page_texts.append(page_text.strip())
    return page_texts

def save_page_texts(page_texts, output_dir, pdf_name):
    pdf_output_dir = output_dir / pdf_name
    pdf_output_dir.mkdir(parents=True, exist_ok=True)
    for page_num, page_text in enumerate(page_texts, start=1):
        page_filename = f"page_{page_num:03d}.txt"
        page_file_path = pdf_output_dir / page_filename
        with open(page_file_path, 'w', encoding='utf-8') as f:
            f.write(page_text)

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    text = re.sub(r'\n\n+', '\n\n', text)
    return text.strip()

def clean_and_save_texts(raw_text_dir, cleaned_text_dir, pdf_name):
    raw_pdf_dir = raw_text_dir / pdf_name
    cleaned_pdf_dir = cleaned_text_dir / pdf_name
    cleaned_pdf_dir.mkdir(parents=True, exist_ok=True)
    for page_file in sorted(raw_pdf_dir.glob("page_*.txt")):
        with open(page_file, 'r', encoding='utf-8') as f:
            raw_text = f.read()
        cleaned_text = clean_text(raw_text)
        output_file = cleaned_pdf_dir / page_file.name
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(cleaned_text)

print("✓ Text extraction and cleaning functions defined")

## 10. Main Processing Pipeline

In [ ]:
def process_single_pdf_complete(
    pdf_path: Path,
    docai_client,
    storage_client,
    processor_name: str,
    bucket_name: str,
    gcs_input_prefix: str,
    gcs_output_prefix: str,
    filtered_pdfs_dir: Path,
    raw_text_dir: Path,
    cleaned_text_dir: Path,
    checkpoint: CheckpointManager,
    confidence_threshold: float
) -> Tuple[str, int, dict]:
    """
    Complete pipeline: CNN filtering → Document AI processing → Text extraction.

    Returns:
        Tuple of (status, extracted_page_count, filtering_stats)
    """
    pdf_name = pdf_path.stem
    start_time = time.time()

    try:
        # STEP 1: CNN Filtering
        print(f"\n{'='*70}")
        print(f"📖 Processing: {pdf_name}")
        print(f"{'='*70}")
        print(f"\nSTEP 1: CNN Filtering")

        filtered_pdf_path = filtered_pdfs_dir / f"{pdf_name}_filtered.pdf"
        filtering_stats = filter_pdf_pages(
            input_pdf_path=pdf_path,
            output_pdf_path=filtered_pdf_path,
            confidence_threshold=confidence_threshold
        )

        # Update checkpoint with filtering stats
        checkpoint.mark_filtering_complete(pdf_name, filtering_stats)

        # If no content pages, skip Document AI processing
        if filtering_stats['content_pages'] == 0:
            print(f"\n⚠️  No content pages found - skipping Document AI processing")
            checkpoint.mark_success(pdf_name, 0)
            return "success", 0, filtering_stats

        # STEP 2: Upload filtered PDF to GCS
        print(f"\nSTEP 2: Uploading filtered PDF to GCS")
        gcs_input_uri = upload_pdf_to_gcs(
            storage_client=storage_client,
            bucket_name=bucket_name,
            pdf_path=filtered_pdf_path,
            gcs_prefix=gcs_input_prefix
        )
        print(f"  ✓ Uploaded: {gcs_input_uri}")

        gcs_output_uri = f"gs://{bucket_name}/{gcs_output_prefix}{pdf_name}/"
        checkpoint.mark_processing(pdf_name, gcs_input_uri, gcs_output_uri)

        # STEP 3: Document AI Batch Processing
        print(f"\nSTEP 3: Document AI Batch Processing")
        operation = submit_batch_process_request(
            client=docai_client,
            processor_name=processor_name,
            gcs_input_uri=gcs_input_uri,
            gcs_output_uri=gcs_output_uri
        )
        print(f"  ✓ Batch request submitted")

        success = wait_for_operation(operation, POLL_INTERVAL, MAX_POLL_ATTEMPTS)
        if not success:
            raise Exception("Document AI processing failed or timed out")

        # STEP 4: Download and parse results
        print(f"\nSTEP 4: Downloading and parsing results")
        result_bytes_list = download_results_from_gcs(storage_client, gcs_output_uri)

        if not result_bytes_list:
            raise Exception("No results found in GCS")

        print(f"  ✓ Downloaded {len(result_bytes_list)} result file(s)")

        # STEP 5: Extract and save text
        print(f"\nSTEP 5: Extracting and saving text")
        all_page_texts = []
        for result_bytes in result_bytes_list:
            document = parse_batch_results(result_bytes)
            page_texts = extract_text_by_page(document)
            all_page_texts.extend(page_texts)

        extracted_page_count = len(all_page_texts)
        print(f"  ✓ Extracted {extracted_page_count} pages")

        # Save raw text
        save_page_texts(all_page_texts, raw_text_dir, pdf_name)
        print(f"  ✓ Saved raw text")

        # Clean and save cleaned text
        clean_and_save_texts(raw_text_dir, cleaned_text_dir, pdf_name)
        print(f"  ✓ Saved cleaned text")

        # STEP 6: Cleanup GCS
        print(f"\nSTEP 6: Cleaning up GCS")
        delete_from_gcs(storage_client, gcs_input_uri)
        delete_from_gcs(storage_client, gcs_output_uri)
        print(f"  ✓ Deleted temporary files from GCS")

        # Update checkpoint
        checkpoint.mark_success(pdf_name, extracted_page_count)

        total_time = time.time() - start_time
        print(f"\n{'='*70}")
        print(f"✅ SUCCESS: {pdf_name}")
        print(f"   Processing time: {total_time:.2f}s ({total_time/60:.2f} min)")
        print(f"   Pages: {filtering_stats['total_pages']} → {filtering_stats['content_pages']} → {extracted_page_count} extracted")
        print(f"{'='*70}")

        return "success", extracted_page_count, filtering_stats

    except Exception as e:
        error_msg = str(e)
        print(f"\n❌ ERROR: {error_msg}")
        checkpoint.mark_failed(pdf_name, error_msg)

        # Try cleanup
        try:
            if 'gcs_input_uri' in locals():
                delete_from_gcs(storage_client, gcs_input_uri)
            if 'gcs_output_uri' in locals():
                delete_from_gcs(storage_client, gcs_output_uri)
        except:
            pass

        return "failed", 0, filtering_stats if 'filtering_stats' in locals() else {}


print("✓ Main processing pipeline defined")

## 11. Run Complete Pipeline

In [ ]:
# Pre-flight checks
print("Pre-flight checks:")
print(f"  PDF folder exists: {PDF_FOLDER.exists()}")
pdf_files = list(PDF_FOLDER.rglob('*.pdf'))
print(f"  PDF count: {len(pdf_files)}")
print(f"  CNN model loaded: {page_classifier_model is not None}")
print(f"  Credentials exist: {CREDENTIALS_PATH.exists()}")
print(f"  GCS bucket ready: gs://{GCS_BUCKET_NAME}")
print(f"\nCheckpoint status:")
print(checkpoint.get_summary())

In [ ]:
# Build processor name
PROCESSOR_NAME = get_processor_name(PROJECT_ID, LOCATION, PROCESSOR_ID)
print(f"Full processor name: {PROCESSOR_NAME}")

In [ ]:
# Get list of PDFs to process
pdf_files = sorted(PDF_FOLDER.rglob("*.pdf"))
pdfs_to_process = [pdf for pdf in pdf_files if not checkpoint.is_processed(pdf.stem)]

print(f"\n📚 Total PDFs found: {len(pdf_files)}")
print(f"📚 Already processed: {len(pdf_files) - len(pdfs_to_process)}")
print(f"📚 To process: {len(pdfs_to_process)}\n")

if pdfs_to_process:
    print("PDFs to process:")
    for i, pdf in enumerate(pdfs_to_process[:10], 1):
        print(f"  {i}. {pdf.name}")
    if len(pdfs_to_process) > 10:
        print(f"  ... and {len(pdfs_to_process) - 10} more")

In [ ]:
# Process all PDFs
if pdfs_to_process:
    print("\n" + "="*70)
    print("STARTING CNN FILTERED + DOCUMENT AI PIPELINE")
    print("="*70 + "\n")

    all_filtering_stats = []
    successful = 0
    failed = 0

    for idx, pdf_path in enumerate(pdfs_to_process, 1):
        print(f"\n[{idx}/{len(pdfs_to_process)}] {pdf_path.name}")

        status, extracted_pages, filtering_stats = process_single_pdf_complete(
            pdf_path=pdf_path,
            docai_client=docai_client,
            storage_client=storage_client,
            processor_name=PROCESSOR_NAME,
            bucket_name=GCS_BUCKET_NAME,
            gcs_input_prefix=GCS_INPUT_PREFIX,
            gcs_output_prefix=GCS_OUTPUT_PREFIX,
            filtered_pdfs_dir=FILTERED_PDFS_DIR,
            raw_text_dir=RAW_TEXT_DIR,
            cleaned_text_dir=CLEANED_TEXT_DIR,
            checkpoint=checkpoint,
            confidence_threshold=CNN_CONFIDENCE_THRESHOLD
        )

        if filtering_stats:
            all_filtering_stats.append(filtering_stats)

        if status == "success":
            successful += 1
        else:
            failed += 1

        print(f"\nProgress: {successful} successful, {failed} failed, {len(pdfs_to_process) - idx} remaining")

    # Save all filtering statistics
    with open(FILTERING_STATS_FILE, 'w', encoding='utf-8') as f:
        json.dump(all_filtering_stats, f, indent=2, ensure_ascii=False)

    # Final summary
    print("\n" + "="*70)
    print("PIPELINE COMPLETE")
    print("="*70)
    print(f"\nProcessing Summary:")
    print(f"  Total PDFs: {len(pdfs_to_process)}")
    print(f"  Successful: {successful}")
    print(f"  Failed: {failed}")

    if all_filtering_stats:
        total_original = sum(s['total_pages'] for s in all_filtering_stats)
        total_content = sum(s['content_pages'] for s in all_filtering_stats)
        total_filtered = sum(s['filtered_pages'] for s in all_filtering_stats)

        print(f"\nFiltering Summary:")
        print(f"  Original pages: {total_original}")
        print(f"  Content pages: {total_content} ({total_content/total_original*100:.1f}%)")
        print(f"  Filtered out: {total_filtered} ({total_filtered/total_original*100:.1f}%)")
        print(f"\n  💰 Savings: Avoided processing {total_filtered} non-content pages!")

    print(f"\nDetailed stats saved to: {FILTERING_STATS_FILE}")


STARTING CNN FILTERED + DOCUMENT AI PIPELINE


[1/64] අංගුත්තර_නිකාය_1.pdf

📖 Processing: අංගුත්තර_නිකාය_1

STEP 1: CNN Filtering

🔍 Filtering: අංගුත්තර_නිකාය_1
📄 Total pages: 661
🤖 Classifying pages with CNN model...



Classifying pages:   0%|          | 0/661 [00:00<?, ?it/s]


✅ Filtered PDF saved: අංගුත්තර_නිකාය_1_filtered.pdf

📊 Filtering Results:
   Total pages: 661
   Content pages: 367 (55.5%)
   Filtered pages: 294 (44.5%)

STEP 2: Uploading filtered PDF to GCS
  ✓ Uploaded: gs://thematic-runner-453210-s8-docai-batch/input_pdfs/අංගුත්තර_නිකාය_1_filtered.pdf

STEP 3: Document AI Batch Processing
  ✓ Batch request submitted
  Waiting for processing to complete...
  Still processing... (150s elapsed)
  ✓ Operation completed successfully

STEP 4: Downloading and parsing results
  ✓ Downloaded 37 result file(s)

STEP 5: Extracting and saving text
  ✓ Extracted 367 pages
  ✓ Saved raw text
  ✓ Saved cleaned text

STEP 6: Cleaning up GCS
  ✓ Deleted temporary files from GCS

✅ SUCCESS: අංගුත්තර_නිකාය_1
   Processing time: 586.66s (9.78 min)
   Pages: 661 → 367 → 367 extracted

Progress: 1 successful, 0 failed, 63 remaining

[2/64] අංගුත්තර_නිකාය_2.pdf

📖 Processing: අංගුත්තර_නිකාය_2

STEP 1: CNN Filtering

🔍 Filtering: අංගුත්තර_නිකාය_2
📄 Total pages: 575
🤖 C

Classifying pages:   0%|          | 0/575 [00:00<?, ?it/s]


✅ Filtered PDF saved: අංගුත්තර_නිකාය_2_filtered.pdf

📊 Filtering Results:
   Total pages: 575
   Content pages: 308 (53.6%)
   Filtered pages: 267 (46.4%)

STEP 2: Uploading filtered PDF to GCS
  ✓ Uploaded: gs://thematic-runner-453210-s8-docai-batch/input_pdfs/අංගුත්තර_නිකාය_2_filtered.pdf

STEP 3: Document AI Batch Processing
  ✓ Batch request submitted
  Waiting for processing to complete...
  Still processing... (150s elapsed)
  ✓ Operation completed successfully

STEP 4: Downloading and parsing results
  ✓ Downloaded 62 result file(s)

STEP 5: Extracting and saving text
  ✓ Extracted 616 pages
  ✓ Saved raw text
  ✓ Saved cleaned text

STEP 6: Cleaning up GCS
  ✓ Deleted temporary files from GCS

✅ SUCCESS: අංගුත්තර_නිකාය_2
   Processing time: 560.89s (9.35 min)
   Pages: 575 → 308 → 616 extracted

Progress: 2 successful, 0 failed, 62 remaining

[3/64] අංගුත්තර_නිකාය_3.pdf

📖 Processing: අංගුත්තර_නිකාය_3

STEP 1: CNN Filtering

🔍 Filtering: අංගුත්තර_නිකාය_3
📄 Total pages: 500
🤖 C

...

Classifying pages:   0%|          | 0/620 [00:00<?, ?it/s]


✅ Filtered PDF saved: ඛුද්දක_පාඨ_filtered.pdf

📊 Filtering Results:
   Total pages: 620
   Content pages: 206 (33.2%)
   Filtered pages: 414 (66.8%)

STEP 2: Uploading filtered PDF to GCS
  ✓ Uploaded: gs://thematic-runner-453210-s8-docai-batch/input_pdfs/ඛුද්දක_පාඨ_filtered.pdf

STEP 3: Document AI Batch Processing
  ✓ Batch request submitted
  Waiting for processing to complete...
  Still processing... (150s elapsed)
  ✓ Operation completed successfully

STEP 4: Downloading and parsing results
  ✓ Downloaded 21 result file(s)

STEP 5: Extracting and saving text
  ✓ Extracted 206 pages
  ✓ Saved raw text
  ✓ Saved cleaned text

STEP 6: Cleaning up GCS
  ✓ Deleted temporary files from GCS

✅ SUCCESS: ඛුද්දක_පාඨ
   Processing time: 467.43s (7.79 min)
   Pages: 620 → 206 → 206 extracted

Progress: 10 successful, 0 failed, 54 remaining

[11/64] ග්‍රන්ථ_අංක_01.pdf

📖 Processing: ග්‍රන්ථ_අංක_01

STEP 1: CNN Filtering

🔍 Filtering: ග්‍රන්ථ_අංක_01
📄 Total pages: 485
🤖 Classifying pages with 

Classifying pages:   0%|          | 0/485 [00:00<?, ?it/s]


✅ Filtered PDF saved: ග්‍රන්ථ_අංක_01_filtered.pdf

📊 Filtering Results:
   Total pages: 485
   Content pages: 114 (23.5%)
   Filtered pages: 371 (76.5%)

STEP 2: Uploading filtered PDF to GCS
  ✓ Uploaded: gs://thematic-runner-453210-s8-docai-batch/input_pdfs/ග්‍රන්ථ_අංක_01_filtered.pdf

STEP 3: Document AI Batch Processing
  ✓ Batch request submitted
  Waiting for processing to complete...
  ✓ Operation completed successfully

STEP 4: Downloading and parsing results
  ✓ Downloaded 12 result file(s)

STEP 5: Extracting and saving text
  ✓ Extracted 114 pages
  ✓ Saved raw text
  ✓ Saved cleaned text

STEP 6: Cleaning up GCS
  ✓ Deleted temporary files from GCS

✅ SUCCESS: ග්‍රන්ථ_අංක_01
   Processing time: 359.33s (5.99 min)
   Pages: 485 → 114 → 114 extracted

Progress: 11 successful, 0 failed, 53 remaining

[12/64] ග්‍රන්ථ_අංක_02.pdf

📖 Processing: ග්‍රන්ථ_අංක_02

STEP 1: CNN Filtering

🔍 Filtering: ග්‍රන්ථ_අංක_02
📄 Total pages: 395
🤖 Classifying pages with CNN model...



Classifying pages:   0%|          | 0/395 [00:00<?, ?it/s]

## 12. View Results and Statistics

In [ ]:
# Display checkpoint summary
print(checkpoint.get_summary())

In [ ]:
# Load and display filtering statistics
if FILTERING_STATS_FILE.exists():
    with open(FILTERING_STATS_FILE, 'r', encoding='utf-8') as f:
        filtering_stats = json.load(f)

    df = pd.DataFrame([{
        'PDF Name': s['pdf_name'],
        'Total Pages': s['total_pages'],
        'Content Pages': s['content_pages'],
        'Filtered Pages': s['filtered_pages'],
        'Content %': f"{s['content_pages']/s['total_pages']*100:.1f}%"
    } for s in filtering_stats])

    print("\n" + "="*70)
    print("CNN FILTERING STATISTICS")
    print("="*70)
    print(df.to_string(index=False))
    print("\nTotals:")
    print(f"  Total PDFs: {len(df)}")
    print(f"  Total Original Pages: {df['Total Pages'].sum()}")
    print(f"  Total Content Pages: {df['Content Pages'].sum()}")
    print(f"  Total Filtered Pages: {df['Filtered Pages'].sum()}")
else:
    print("No filtering statistics available yet.")

In [ ]:
# Sample extracted text
sample_pdfs = sorted([d for d in RAW_TEXT_DIR.iterdir() if d.is_dir()])
if sample_pdfs:
    sample_pdf_dir = sample_pdfs[0]
    sample_pages = list(sample_pdf_dir.glob("page_*.txt"))

    if sample_pages:
        sample_page = sample_pages[0]

        print("\n" + "="*60)
        print(f"SAMPLE EXTRACTED TEXT")
        print(f"PDF: {sample_pdf_dir.name}")
        print(f"Page: {sample_page.name}")
        print("="*60)

        with open(sample_page, 'r', encoding='utf-8') as f:
            sample_text = f.read()

        print(sample_text[:500])
        print("\n[...truncated...]" if len(sample_text) > 500 else "")

## Summary

**This notebook combines:**
1. ✓ CNN-based page classification (your trained model)
2. ✓ PDF filtering (creates new PDFs with only content pages)
3. ✓ Document AI batch processing (processes filtered PDFs)
4. ✓ Text extraction and cleaning
5. ✓ Comprehensive statistics and checkpointing

**Key Benefits:**
- **Cost savings**: Only process content pages with Document AI
- **Quality**: Remove covers, blank pages, and non-content automatically
- **Transparency**: See exactly how many pages were filtered per PDF
- **Resume capability**: Checkpoint system allows resuming if interrupted

**Output structure:**
```
data/
├── filtered_pdfs/           # CNN-filtered PDFs (only content pages)
├── docai_extractions/
│   ├── 1_raw_text/         # Extracted text per page
│   ├── 2_cleaned_text/     # Cleaned text per page
│   └── logs/
│       ├── processing_checkpoint_filtered.json
│       └── filtering_stats.json
```

**Statistics tracked:**
- Pages before filtering
- Pages after filtering
- Number filtered out
- Percentage of content vs. non-content
- Per-page classification details